# Evaluating model outputs: accuracy, precision, recall, F1, and the confusion matrix

This is the hands-on companion to the accuracy, precision, recall, F1, and confusion matrix lesson earlier this week. If you haven't read that yet, start there — this notebook assumes those definitions are already familiar, and walks through running them for real.

**Time**: ~20 minutes
**Cost**: A few cents at most (we'll estimate before we run anything — see Session 1's token cost guide)

> **Running this locally, or in VS Code instead of Colab?** See [Session 1's setup guide](../session_1/setup_guide.ipynb) for how to open any of these notebooks with `uv`, either in a browser tab or inside VS Code. Nothing extra to do if you're in Colab.

## What this template does

An API call succeeding tells you the plumbing worked; it says nothing about whether the answer is right. This template closes that gap directly: it compares a model's predictions against a set of **known correct answers** (called *ground truth* or *true labels*) and reports exactly how often, and in what way, it gets things wrong.

## Setup

If you completed the Session 1 setup guide, your Gemini API key is already saved in Colab Secrets — there's nothing extra to do here. If you haven't, go do that first: open `setup_guide.ipynb` in the `session_1` folder of the course repository.

**In Colab**: nothing to do beforehand, the cell below installs everything needed.

**Locally (VS Code, PyCharm, or any other IDE)**: run this once in a terminal, before opening this notebook, to create this session's own environment:

```bash
cd session_2
uv venv
uv pip install -r requirements.txt
```

Then point this notebook's kernel at `session_2/.venv/bin/python` (VS Code: Select Kernel → Python Environments; PyCharm: Settings → Project → Python Interpreter → Add Interpreter → Existing). Full walkthrough in [Session 1's setup guide](../session_1/setup_guide.ipynb).

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab: installs from an explicit list, not session_2/requirements.txt --
    # "Open in Colab" only loads this one file, so there's no repo alongside it
    # to read that file from. This list is that file's contents written out
    # directly; keep the two in sync if you add or remove a package.
    %pip install -q google-genai pandas numpy matplotlib seaborn scikit-learn nltk rouge-score
else:
    # Local: installs from session_2/requirements.txt. Looked up by directory,
    # not just filename, since the repo also has its own top-level
    # requirements.txt for whole-repo setup -- a same-named but wrong file a
    # plain filename search could grab by mistake from the repo root.
    import pathlib
    _cwd = pathlib.Path.cwd()
    if _cwd.name == "session_2" and (_cwd / "requirements.txt").exists():
        _req_path = "requirements.txt"
    elif (_cwd / "session_2" / "requirements.txt").exists():
        _req_path = "session_2/requirements.txt"
    elif (_cwd.parent / "session_2" / "requirements.txt").exists():
        _req_path = "../session_2/requirements.txt"
    else:
        _req_path = None

    if _req_path is None:
        print("Couldn't find session_2/requirements.txt from the current working directory:", _cwd)
        print("Set up your local environment first -- see the markdown cell above for the uv commands.")
    else:
        %pip install -q -r {_req_path}

In [ ]:
from google import genai
from google.colab import userdata

# Config: change this one line to point the whole notebook at a different Gemini model.
# Swapping to a different provider (OpenAI, Anthropic, ...) means swapping this client
# and the call inside classify() below — see Session 5 for a provider-agnostic pattern.
MODEL_NAME = "gemini-3.5-flash-lite"

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
print("Connected.")

## The evaluation dataset

To keep this notebook self-contained and privacy-safe, we're using a small **synthetic** dataset — 45 invented customer feedback snippets, each hand-labeled with one of five topics an agency or CX team would recognize. No real customer data, nothing to anonymize.

**Swap this out** for your own labeled data later — anything with a text column and a true-label column works with the rest of this notebook.

In [ ]:
import pandas as pd

CATEGORIES = [
    "Shipping & Delivery",
    "Product Quality",
    "Pricing & Billing",
    "Customer Service",
    "Returns & Refunds",
]

data = [
    ("My package arrived five days later than the estimated delivery date.", "Shipping & Delivery"),
    ("Tracking said it was out for delivery but it never showed up.", "Shipping & Delivery"),
    ("Shipping was actually really fast, got it in two days!", "Shipping & Delivery"),
    ("The box arrived completely crushed, I think it was thrown around.", "Shipping & Delivery"),
    ("Delivery driver left it in the rain even though I asked for it to go behind the gate.", "Shipping & Delivery"),
    ("I paid for express shipping but it still took a week.", "Shipping & Delivery"),
    ("Great communication throughout, I always knew where my order was.", "Shipping & Delivery"),
    ("Wrong address on the label caused a huge delay.", "Shipping & Delivery"),
    ("The courier was really friendly and delivered right on time.", "Shipping & Delivery"),

    ("The fabric started pilling after just one wash.", "Product Quality"),
    ("Way better quality than I expected for the price.", "Product Quality"),
    ("The zipper broke on the second use.", "Product Quality"),
    ("Solid build, feels like it'll last for years.", "Product Quality"),
    ("Colors looked nothing like the photos online.", "Product Quality"),
    ("Stitching came undone within a week.", "Product Quality"),
    ("This is by far the best version of this product I've owned.", "Product Quality"),
    ("The material feels cheap and flimsy.", "Product Quality"),
    ("Exceeded my expectations, very well made.", "Product Quality"),

    ("I was charged twice for the same order.", "Pricing & Billing"),
    ("The price displayed at checkout didn't match what was on my card statement.", "Pricing & Billing"),
    ("Great value for the price, would buy again.", "Pricing & Billing"),
    ("Subscription renewed without any warning and charged my card.", "Pricing & Billing"),
    ("There was a hidden fee I wasn't told about until checkout.", "Pricing & Billing"),
    ("Prices are way too high compared to competitors.", "Pricing & Billing"),
    ("Refund for the price difference was processed quickly.", "Pricing & Billing"),
    ("I appreciate the transparent pricing, no surprises.", "Pricing & Billing"),
    ("My discount code didn't apply and I paid full price.", "Pricing & Billing"),

    ("The support agent was incredibly patient and solved my issue in minutes.", "Customer Service"),
    ("I waited on hold for over an hour and never got through.", "Customer Service"),
    ("Nobody responded to my email for a week.", "Customer Service"),
    ("The chat agent was rude and dismissive.", "Customer Service"),
    ("They went above and beyond to fix my problem.", "Customer Service"),
    ("I had to explain my issue three times to three different agents.", "Customer Service"),
    ("Quick, friendly, and knowledgeable support team.", "Customer Service"),
    ("Support kept transferring me in circles.", "Customer Service"),
    ("Really appreciated the follow-up call to make sure everything was resolved.", "Customer Service"),

    ("My return was processed within two days, very smooth.", "Returns & Refunds"),
    ("I've been waiting three weeks for my refund and still nothing.", "Returns & Refunds"),
    ("The return label they sent didn't work at the post office.", "Returns & Refunds"),
    ("Easiest return process I've ever dealt with.", "Returns & Refunds"),
    ("They refused to refund me even though the item was defective.", "Returns & Refunds"),
    ("Refund showed up on my card exactly as promised.", "Returns & Refunds"),
    ("Had to pay for return shipping even though it was their mistake.", "Returns & Refunds"),
    ("Customer service made the return painless.", "Returns & Refunds"),
    ("Still waiting on a refund from a return I sent back a month ago.", "Returns & Refunds"),
]

df = pd.DataFrame(data, columns=["text", "true_label"])
print(f"{len(df)} labeled examples across {df['true_label'].nunique()} categories")
df.sample(5, random_state=1)

## Generate predictions

Now let's have Gemini classify each snippet, without telling it the true label, so we have something to evaluate. This is the same pattern as item 4's "first AI assistant use case" — classification — just on our placeholder dataset instead of a live one.

**Before running a batch job, estimate the cost** (habit from the token cost guide). 45 short classification calls on `gemini-3.5-flash-lite` costs a fraction of a cent — but there's no free quota behind it anymore, so this is a good moment to check that habit even when the dollar amount is tiny.

In [ ]:
import time
from google.genai import types

CATEGORY_LIST = ", ".join(CATEGORIES)

def classify(text: str, retries: int = 2) -> str:
    prompt = f"""Classify the following customer feedback into exactly ONE of these categories:
{CATEGORY_LIST}

Feedback: "{text}"

Respond with only the category name, exactly as written above. Nothing else."""

    for attempt in range(retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                # temperature=0 makes the model as deterministic as it can be, so re-running
                # this notebook gives you (close to) the same predictions and metrics each time.
                config=types.GenerateContentConfig(temperature=0),
            )
            label = response.text.strip()
            # Guard against the model returning something slightly off-format
            return label if label in CATEGORIES else "UNKNOWN"
        except Exception as e:
            if attempt == retries:
                print(f"  Giving up on one row after {retries} retries ({e}) — recording as UNKNOWN.")
                return "UNKNOWN"
            time.sleep(2)

predictions = [classify(text) for text in df["text"]]
df["predicted_label"] = predictions

# UNKNOWN covers both off-format replies and calls that failed every retry. It's a real
# outcome the model produced, so it's kept as its own label below (rather than silently
# dropped) -- otherwise accuracy and the confusion matrix would disagree with each other.
LABELS = CATEGORIES + ["UNKNOWN"] if "UNKNOWN" in df["predicted_label"].values else CATEGORIES

unknown_count = (df["predicted_label"] == "UNKNOWN").sum()
if unknown_count:
    print(f"⚠️ {unknown_count} response(s) didn't match a known category exactly (or failed every retry) — worth showing the class as a real-world formatting failure mode.")
print("Done.")
df.head()

## Accuracy

As covered in the lesson, accuracy alone can be misleading on imbalanced data — a model that always guesses the majority category can outscore one that's actually learned something. This notebook's own dataset is deliberately balanced at nine items per category, so running it here won't demonstrate that effect; that's a property of this sample data, not a sign anything's wrong with the code below. Real customer feedback is almost never this evenly split.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(df["true_label"], df["predicted_label"])
print(f"Overall accuracy: {accuracy:.1%}  ({int(accuracy * len(df))} of {len(df)} correct)")

## The confusion matrix

The matrix below follows the same read as the lesson: rows are the true label, columns are the prediction, the diagonal is everything the model got right. One detail specific to this notebook: a reply that doesn't match one of the five category names, or a call that fails outright after retrying, is recorded as its own `UNKNOWN` category rather than dropped, so it shows up as a distinct row and column here instead of silently vanishing from the count.

One note on axes: scikit-learn's `confusion_matrix`, used below, puts true labels on the rows and predictions on the columns, matching the description above. Some other tools flip this, so check the axis labels before reading anyone else's matrix.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(df["true_label"], df["predicted_label"], labels=LABELS)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Precision, recall, and F1, by category

Precision, recall, and F1 are computed per category below, exactly as defined in the lesson. One detail specific to this notebook: predictions can include the `UNKNOWN` category described above, so it's passed alongside the five real topics into every metric call below — otherwise accuracy and the confusion matrix would disagree about how many rows exist.

In [ ]:
from sklearn.metrics import classification_report

# zero_division=0 controls what to print when a category never appears in the predictions
# (or never appears in the true labels) -- without it, sklearn raises a warning for the
# resulting 0/0 division and asks you to pick a value; 0 is the standard "undefined" choice.
print(classification_report(df["true_label"], df["predicted_label"], labels=LABELS, zero_division=0))

### Verifying F1 against the formula

`classification_report` prints F1 for every category. As a quick check that it's computing what the lesson described, the cell below picks one category and confirms sklearn's F1 matches the harmonic-mean formula applied directly to that category's own precision and recall.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    df["true_label"], df["predicted_label"], labels=LABELS, zero_division=0
)

# Show the arithmetic for one category, and confirm it matches sklearn's F1 above
example_idx = 0
p, r = precision[example_idx], recall[example_idx]
manual_f1 = 2 * (p * r) / (p + r) if (p + r) else 0.0

print(f"Category: {LABELS[example_idx]}")
print(f"  precision = {p:.3f}, recall = {r:.3f}")
print(f"  harmonic mean  2 * (precision * recall) / (precision + recall) = {manual_f1:.3f}")
print(f"  sklearn's F1                                                   = {f1[example_idx]:.3f}")

## The reusable template

Everything above, wrapped into one function. This is the part anyone in the course can copy into their own notebook: swap in your own `true_label` / `predicted_label` columns and run it.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate(y_true, y_pred, labels=None, title="Evaluation Results"):
    """
    Evaluate classification predictions against ground truth.
    Works on ANY labeled classification task -- not just this dataset.

    y_true, y_pred : lists or pandas Series of labels (same length, same order)
    labels         : ordered list of all possible category names (optional --
                      inferred from the data if not given)
    """
    observed = set(y_true) | set(y_pred)
    if labels is None:
        labels = sorted(observed)
    else:
        # Keep any label the caller passed in, but don't silently drop a label that shows
        # up in the data and wasn't listed (e.g. an UNKNOWN/fallback value) -- dropping it
        # would make accuracy and the confusion matrix disagree on how many rows there are.
        labels = list(labels) + sorted(observed - set(labels))

    accuracy = accuracy_score(y_true, y_pred)
    print(f"{title}")
    print("=" * len(title))
    print(f"Overall accuracy: {accuracy:.1%}\n")

    print("Per-category breakdown (precision / recall / F1):")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return {"accuracy": accuracy, "confusion_matrix": cm}

# Try it on our dataset:
results = evaluate(df["true_label"], df["predicted_label"], labels=CATEGORIES, title="Topic Classification Evaluation")

## Using this on your own data

To evaluate your own model outputs instead of this placeholder dataset:

1. Get your data into a table with (at minimum) a `true_label` column and a `predicted_label` column — same row order, same category names in both.
2. If you're starting from a CSV: `df = pd.read_csv('your_file.csv')`
3. Run: `evaluate(df['true_label'], df['predicted_label'], labels=[...your categories...])`

That's it — the function doesn't care what the categories are or where the predictions came from (Gemini, another model, or even a human).

## Wrap-up

The lesson explained what these numbers mean; this notebook is what actually produces them, against a real (if synthetic) dataset, ready to swap for real data whenever it's needed.

**Next in Session 2**: Human-in-the-loop validation — at what point does a wrong prediction actually reach a business decision, and where does a human need to check it first? The per-category breakdown above is exactly the tool you'd use to decide that.

**Questions?** Post in the Circle community.